In [97]:
import chromadb


from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from typing import List

In [98]:
loader = PyPDFLoader("https://www.gita-society.com/bhagavad-gita-in-english-source-file.pdf")
# loader = PyPDFLoader("https://www.prabhupada-books.de/pdf/Bhagavad-gita-As-It-Is.pdf")

document = loader.load()

In [99]:
document

[Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2021-08-05T09:35:03-07:00', 'title': 'Bhagavad-Gita verses in simple english Language', 'author': 'Ram', 'moddate': '2021-08-05T09:35:03-07:00', 'source': 'https://www.gita-society.com/bhagavad-gita-in-english-source-file.pdf', 'total_pages': 53, 'page': 0, 'page_label': '1'}, page_content='BHAGAVAD-GITA in ENGLISH (Source) \nFor commentaries: \nhttps://www.gita-society.com/Read-bhagavad-gita.html \n \nCONTENTS \nINTRODUCTION ...................................................... 1 \n1. Arjuna’s Dilemma .................................................. 3 \n2. Spiritual knowledge................................................ 3 \nThe spirit is eternal, body is transitory ........................ 4 \nDeath and Reincarnation of the soul .......................... 4 \nDuty of a warrior ......................................................... 5 \nImportance of Karma-yoga, the selfl

In [ ]:
from typing import Any

class EmbeddingManager:
    def __init__(self, model_name:'str'):
        """
        Intialze tne embedding manager
        
        Args: 
            model_name: hugging face model for sentense enbedding
        """
        self.model_name = model_name
        self.model = None
        self._load_model()
        
    def _load_model(self):
        """ Loader function to load the transformer Model """
        try: 
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Embedding model loaded successfully with dimenstions: {self.model.get_embedding_dimension()}")
        except Exception as e: 
            print(f"Failed to load model {self.model_name}", e)
            raise  
          
    def generate_embeddings(self, text: Any):
        if not self.model: 
            raise ValueError("Model is not loaded")
        print(f"generating embeddings for {len(text)} text")
        embeddings = self.model.encode(text, show_progress_bar=False)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings  
    
    def document_chunker(self, document_chunk: Any):
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=100,
            chunk_overlap=0
            )
        return text_splitter.split_text(document_chunk)

In [101]:
# Initialize embedding manager
embedding_model = "all-MiniLM-L6-v2"
embedding_manager = EmbeddingManager(embedding_model)
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3220.83it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded successfully with dimenstions: 384


In [102]:
# Chunk the document
text_chunks = []
embeddings = []
for item in document:
    chunks = embedding_manager.document_chunker(item.page_content)
    item = {
        "text": chunks,
        "metas": item.metadata
    }
    text_chunks.append(item)

# Generate Embeddings
   
for chunk in text_chunks:
    embeddings.append(embedding_manager.generate_embeddings(chunk["text"]))    

generating embeddings for 34 text
Generated embeddings with shape: (34, 384)
generating embeddings for 42 text
Generated embeddings with shape: (42, 384)
generating embeddings for 42 text
Generated embeddings with shape: (42, 384)
generating embeddings for 28 text
Generated embeddings with shape: (28, 384)
generating embeddings for 35 text
Generated embeddings with shape: (35, 384)
generating embeddings for 44 text
Generated embeddings with shape: (44, 384)
generating embeddings for 28 text
Generated embeddings with shape: (28, 384)
generating embeddings for 45 text
Generated embeddings with shape: (45, 384)
generating embeddings for 41 text
Generated embeddings with shape: (41, 384)
generating embeddings for 42 text
Generated embeddings with shape: (42, 384)
generating embeddings for 41 text
Generated embeddings with shape: (41, 384)
generating embeddings for 41 text
Generated embeddings with shape: (41, 384)
generating embeddings for 44 text
Generated embeddings with shape: (44, 384)

In [103]:
embeddings

[array([[-0.06078739, -0.03106729, -0.10074292, ...,  0.09641363,
         -0.03104571,  0.02565662],
        [-0.00137   ,  0.03884903, -0.0615646 , ...,  0.07351412,
         -0.01735491,  0.04847431],
        [-0.04368674,  0.04568758,  0.00733063, ...,  0.12954451,
         -0.00709394,  0.0534982 ],
        ...,
        [-0.02585353,  0.03530272, -0.02947188, ..., -0.01104341,
         -0.04425392, -0.02750926],
        [ 0.01460298,  0.07125333,  0.0762705 , ...,  0.05111398,
         -0.01637856, -0.03523138],
        [ 0.02651373,  0.03040301, -0.01994872, ..., -0.06826366,
          0.04187539,  0.00118987]], shape=(34, 384), dtype=float32),
 array([[-0.0014406 , -0.02839266, -0.06773519, ..., -0.07346319,
         -0.0198315 , -0.01851532],
        [-0.09807996,  0.0814175 ,  0.03701157, ...,  0.01530297,
         -0.02146495, -0.03031926],
        [-0.00646636,  0.08492517, -0.02739456, ...,  0.02213583,
         -0.02983858,  0.01821707],
        ...,
        [ 0.01498399, 

In [104]:
collection_name = 'bhagwat_gita_plain_english'
persistant_directory = '/home/kapil/rag-from-scratch/vector_store'

In [105]:
vector_store = chromadb.PersistentClient(
    # collection_name=collection_name,
    # embedding_function="all-MiniLM-L6-v2",
    path="/home/kapil/rag-from-scratch/vector_store"
)

vector_store.get_or_create_collection(
    name="bhagwat_gita_plain_english",
    # embedding_function=SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2"),
    metadata={
        "description": "my first Chroma collection"    
        }
    )

Collection(name=bhagwat_gita_plain_english)

In [111]:
collections = vector_store.list_collections()

collections


[Collection(name=bhagwat_gita_plain_english)]

In [107]:
collection = vector_store.get_collection("bhagwat_gita_plain_english")

collection

Collection(name=bhagwat_gita_plain_english)

In [ ]:
ids = [i for i in range(0, 53)]

vector_store

In [ ]:
collection.add(
    ids=ids,
    embeddings=embeddings,
    documents=text_chunks,
    # metadatas=[{"chapter": 3, "verse": 16}, {"chapter": 3, "verse": 5}, {"chapter": 29, "verse": 11}],
)

AttributeError: 'Client' object has no attribute 'add'